# 01 - Data Collection

## Lahore AQI Predictor

This notebook collects historical hourly weather and air-quality data
for Lahore, Pakistan.

### Data Sources

- Weather: Open-Meteo Historical Weather API
- Air Quality: Open-Meteo Air Quality API

### Historical Period

Start Date: 2024-01-01  
End Date: 2026-07-31

### Target

The primary prediction target is:

`us_aqi`

### Collected Weather Features

- temperature_2m
- relative_humidity_2m
- surface_pressure
- precipitation
- cloud_cover
- wind_speed_10m
- wind_direction_10m

### Collected Air Quality Features

- pm2_5
- pm10
- carbon_monoxide
- nitrogen_dioxide
- sulphur_dioxide
- ozone
- us_aqi

In [1]:
import openmeteo_requests
import requests_cache
import pandas as pd

from retry_requests import retry

## 1. Configure Open-Meteo API Client

In [2]:
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=-1
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)

print("Open-Meteo client configured successfully.")

Open-Meteo client configured successfully.


## 2. Load City Configuration

In [8]:
from pathlib import Path

base_dir = Path.cwd()
candidates = [
    base_dir / "data" / "cities.csv",
    base_dir.parent / "data" / "cities.csv",
    base_dir / "config" / "cities.csv",
]
city_path = next((path for path in candidates if path.exists()), base_dir / "data" / "cities.csv")

cities = pd.read_csv(city_path)

lahore = cities[
    cities["city"].str.lower() == "lahore"
].iloc[0]

latitude = float(lahore["latitude"])
longitude = float(lahore["longitude"])

print("City:", lahore["city"])
print("Latitude:", latitude)
print("Longitude:", longitude)

City: Lahore
Latitude: 31.5204
Longitude: 74.3587


## 3. Historical Collection Period

In [9]:
start_date = "2024-01-01"
end_date = "2026-07-31"

print("Start date:", start_date)
print("End date:", end_date)

Start date: 2024-01-01
End date: 2026-07-31


## 4. Collect Historical Weather Data

In [10]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "surface_pressure",
        "precipitation",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m"
    ],
    "timezone": "Asia/Karachi"
}

responses = openmeteo.weather_api(
    url,
    params=params
)

response = responses[0]

print("API response received successfully.")
print("Returned latitude:", response.Latitude())
print("Returned longitude:", response.Longitude())

API response received successfully.
Returned latitude: 31.52899742126465
Returned longitude: 74.38995361328125


### Convert API Response to DataFrame

In [12]:
hourly = response.Hourly()

weather_df = pd.DataFrame({
    "datetime": pd.date_range(
        start=pd.to_datetime(
            hourly.Time(),
            unit="s",
            utc=True
        ).tz_convert("Asia/Karachi").tz_localize(None),

        periods=hourly.Variables(0).ValuesAsNumpy().shape[0],

        freq=pd.Timedelta(
            seconds=hourly.Interval()
        )
    ),

    "temperature_2m":
        hourly.Variables(0).ValuesAsNumpy(),

    "relative_humidity_2m":
        hourly.Variables(1).ValuesAsNumpy(),

    "surface_pressure":
        hourly.Variables(2).ValuesAsNumpy(),

    "precipitation":
        hourly.Variables(3).ValuesAsNumpy(),

    "cloud_cover":
        hourly.Variables(4).ValuesAsNumpy(),

    "wind_speed_10m":
        hourly.Variables(5).ValuesAsNumpy(),

    "wind_direction_10m":
        hourly.Variables(6).ValuesAsNumpy()
})

### Initial Weather Data Inspection

In [13]:
print("Shape:", weather_df.shape)

print("\nFirst records:")
display(weather_df.head())

print("\nLast records:")
display(weather_df.tail())

print("\nData types:")
display(weather_df.dtypes)

Shape: (22632, 8)

First records:


,datetime,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,cloud_cover,wind_speed_10m,wind_direction_10m
0,2024-01-01 00:00:00,6.15,98.969955,992.694580,0.0,100.0,3.415260,161.564957
1,2024-01-01 01:00:00,6.05,98.287132,992.587891,0.0,100.0,2.276840,161.564957
2,2024-01-01 02:00:00,6.35,98.971619,992.616333,0.0,100.0,2.189795,260.537750
3,2024-01-01 03:00:00,6.15,99.655602,992.694580,0.0,100.0,3.096837,305.537750
4,2024-01-01 04:00:00,6.55,97.282684,992.538025,0.0,100.0,3.096837,305.537750



Last records:


,datetime,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,cloud_cover,wind_speed_10m,wind_direction_10m
22627,2026-07-31 19:00:00,29.35,84.930939,974.444702,0.0,100.0,9.292255,58.465260
22628,2026-07-31 20:00:00,29.15,86.171577,975.014160,0.0,100.0,9.165871,70.497467
22629,2026-07-31 21:00:00,28.85,86.909058,975.575378,0.0,98.0,9.001800,78.465408
22630,2026-07-31 22:00:00,28.40,86.611740,976.026917,0.0,88.0,7.922045,88.698082
22631,2026-07-31 23:00:00,28.00,87.608261,976.092224,0.0,73.0,8.731392,81.702942



Data types:


datetime                datetime64[s]
temperature_2m                float32
relative_humidity_2m          float32
surface_pressure              float32
precipitation                 float32
cloud_cover                   float32
wind_speed_10m                float32
wind_direction_10m            float32
dtype: object

### Weather Data Validation

In [14]:
print("Missing values:")
display(weather_df.isna().sum())

print("\nMinimum datetime:")
print(weather_df["datetime"].min())

print("\nMaximum datetime:")
print(weather_df["datetime"].max())

duplicates = weather_df["datetime"].duplicated().sum()

print("\nDuplicate timestamps:", duplicates)

Missing values:


datetime                0
temperature_2m          0
relative_humidity_2m    0
surface_pressure        0
precipitation           0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
dtype: int64


Minimum datetime:
2024-01-01 00:00:00

Maximum datetime:
2026-07-31 23:00:00

Duplicate timestamps: 0


In [15]:
print("Number of rows:", len(weather_df))
print("Unique timestamps:", weather_df["datetime"].nunique())

Number of rows: 22632
Unique timestamps: 22632


## 5. Save Raw Weather Data

In [19]:
from pathlib import Path

output_path = Path("../data/raw/lahore/weather/lahore_weather_hourly.csv")

# Create the folder if it doesn't exist
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save CSV
weather_df.to_csv(output_path, index=False)

print(f"Saved successfully to: {output_path}")

Saved successfully to: ..\data\raw\lahore\weather\lahore_weather_hourly.csv


# 6. Collect Historical Air Quality Data

This section retrieves hourly historical air-quality data for Lahore
using the Open-Meteo Air Quality API.

The collected pollutants include:

- PM2.5
- PM10
- Carbon Monoxide (CO)
- Nitrogen Dioxide (NO2)
- Sulphur Dioxide (SO2)
- Ozone (O3)
- US AQI

In [22]:
# ==========================================
# Open-Meteo Air Quality API
# ==========================================

air_quality_url = "https://air-quality-api.open-meteo.com/v1/air-quality"

air_quality_params = {
    "latitude": latitude,
    "longitude": longitude,

    "start_date": start_date,
    "end_date": end_date,

    "hourly": [
        "pm2_5",
        "pm10",
        "carbon_monoxide",
        "nitrogen_dioxide",
        "sulphur_dioxide",
        "ozone",
        "us_aqi"
    ],

    "timezone": "Asia/Karachi"
}

air_quality_responses = openmeteo.weather_api(
    air_quality_url,
    params=air_quality_params
)

air_quality_response = air_quality_responses[0]

print("Air quality API response received successfully.")
print("City:", lahore["city"])
print("Returned latitude:", air_quality_response.Latitude())
print("Returned longitude:", air_quality_response.Longitude())

Air quality API response received successfully.
City: Lahore
Returned latitude: 31.5
Returned longitude: 74.40000915527344


## 7 Convert Air Quality API Response to DataFrame

In [23]:
# ==========================================
# Extract hourly air quality data
# ==========================================

air_quality_hourly = air_quality_response.Hourly()

pollution_df = pd.DataFrame({

    "datetime": pd.date_range(
        start=pd.to_datetime(
            air_quality_hourly.Time(),
            unit="s",
            utc=True
        ).tz_convert("Asia/Karachi").tz_localize(None),

        periods=air_quality_hourly.Variables(0).ValuesAsNumpy().shape[0],

        freq=pd.Timedelta(
            seconds=air_quality_hourly.Interval()
        )
    ),

    "pm2_5":
        air_quality_hourly.Variables(0).ValuesAsNumpy(),

    "pm10":
        air_quality_hourly.Variables(1).ValuesAsNumpy(),

    "carbon_monoxide":
        air_quality_hourly.Variables(2).ValuesAsNumpy(),

    "nitrogen_dioxide":
        air_quality_hourly.Variables(3).ValuesAsNumpy(),

    "sulphur_dioxide":
        air_quality_hourly.Variables(4).ValuesAsNumpy(),

    "ozone":
        air_quality_hourly.Variables(5).ValuesAsNumpy(),

    "us_aqi":
        air_quality_hourly.Variables(6).ValuesAsNumpy()
})

print("Pollution DataFrame created successfully.")
print("Shape:", pollution_df.shape)

Pollution DataFrame created successfully.
Shape: (22632, 8)


### Initial Air Quality Data Inspection

In [24]:
print("First records:")

display(pollution_df.head())

First records:


,datetime,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi
0,2024-01-01 00:00:00,260.799988,375.299988,4925.0,154.800003,51.500000,11.0,272.108368
1,2024-01-01 01:00:00,260.799988,375.299988,4687.0,147.000000,41.099998,6.0,272.729156
2,2024-01-01 02:00:00,239.300003,344.600006,4326.0,132.199997,32.099998,8.0,273.391632
3,2024-01-01 03:00:00,208.600006,299.000000,3733.0,103.500000,26.400000,24.0,273.429169
4,2024-01-01 04:00:00,196.000000,281.299988,3016.0,67.800003,22.200001,48.0,273.079163


In [25]:
print("Last records:")

display(pollution_df.tail())

Last records:


,datetime,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi
22627,2026-07-31 19:00:00,38.500000,38.799999,817.0,37.099998,9.6,79.0,154.655609
22628,2026-07-31 20:00:00,39.099998,39.299999,888.0,44.099998,10.2,54.0,135.374146
22629,2026-07-31 21:00:00,38.000000,38.200001,858.0,43.400002,10.3,45.0,114.354172
22630,2026-07-31 22:00:00,37.799999,38.200001,773.0,38.799999,10.3,43.0,111.322914
22631,2026-07-31 23:00:00,38.500000,39.000000,683.0,35.200001,10.1,41.0,109.031250


In [26]:
print("Data types:")

display(pollution_df.dtypes)

Data types:


datetime            datetime64[s]
pm2_5                     float32
pm10                      float32
carbon_monoxide           float32
nitrogen_dioxide          float32
sulphur_dioxide           float32
ozone                     float32
us_aqi                    float32
dtype: object

## 8. Validate Historical Air Quality Data

The following checks verify:

1. Missing values
2. Duplicate timestamps
3. Number of rows
4. Number of unique timestamps
5. Date range
6. Basic statistical values

In [27]:
# ==========================================
# Missing values
# ==========================================

print("Missing values:")

display(pollution_df.isna().sum())

Missing values:


datetime            0
pm2_5               0
pm10                0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
us_aqi              0
dtype: int64

In [31]:
# ==========================================
# Row and timestamp validation
# ==========================================

print("Pollution Data rows:", len(pollution_df))

print(
    "Pollution Data unique timestamps:",
    pollution_df["datetime"].nunique()
)

Pollution Data rows: 22632
Pollution Data unique timestamps: 22632


In [32]:
# ==========================================
# Date range validation
# ==========================================

print(
    "Minimum datetime:",
    pollution_df["datetime"].min()
)

print(
    "Maximum datetime:",
    pollution_df["datetime"].max()
)

Minimum datetime: 2024-01-01 00:00:00
Maximum datetime: 2026-07-31 23:00:00


In [33]:
# ==========================================
# Statistical summary
# ==========================================

display(
    pollution_df.describe()
)

,datetime,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi
count,22632,22632.000000,22632.000000,22632.000000,22632.000000,22632.000000,22632.000000,22632.000000
mean,2025-04-16 11:30:00,68.301697,100.604340,1269.717041,38.739552,15.887641,90.933632,151.115616
min,2024-01-01 00:00:00,3.400000,3.600000,153.000000,0.000000,0.100000,0.000000,56.139935
25%,2024-08-23 17:45:00,34.200001,52.200001,492.000000,12.700000,9.600000,38.000000,112.049484
50%,2025-04-16 11:30:00,51.900002,78.699997,823.000000,28.299999,13.000000,75.000000,150.988098
75%,2025-12-08 05:15:00,84.300003,126.699997,1649.250000,57.700001,18.900000,142.000000,175.795391
max,2026-07-31 23:00:00,365.100006,890.299988,8602.000000,201.899994,94.400002,305.000000,537.537537
std,NaN,50.450336,75.164955,1150.900024,33.237381,9.644478,64.009148,50.324200


In [34]:
# ==========================================
# Check for negative pollutant values
# ==========================================

pollutant_columns = [
    "pm2_5",
    "pm10",
    "carbon_monoxide",
    "nitrogen_dioxide",
    "sulphur_dioxide",
    "ozone",
    "us_aqi"
]

negative_values = (
    pollution_df[pollutant_columns] < 0
).sum()

print("Negative values by column:")

display(negative_values)

Negative values by column:


pm2_5               0
pm10                0
carbon_monoxide     0
nitrogen_dioxide    0
sulphur_dioxide     0
ozone               0
us_aqi              0
dtype: int64

In [35]:
# ==========================================
# AQI range
# ==========================================

print("Minimum AQI:", pollution_df["us_aqi"].min())
print("Maximum AQI:", pollution_df["us_aqi"].max())
print("Average AQI:", pollution_df["us_aqi"].mean())

Minimum AQI: 56.139935
Maximum AQI: 537.53754
Average AQI: 151.11562


## 9. Save Raw Air Quality Data

The validated historical air-quality dataset is saved as a raw CSV
for use in subsequent validation, EDA, and feature-engineering stages.

In [37]:
# ==========================================
# Save raw pollution data
# ==========================================

from pathlib import Path

pollution_output_path = Path(
    "../data/raw/lahore/pollution/lahore_pollution_hourly.csv"
)

# Create the directory if it doesn't exist
pollution_output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the CSV file
pollution_df.to_csv(
    pollution_output_path,
    index=False
)

print("Pollution data saved successfully.")
print(pollution_output_path)

Pollution data saved successfully.
..\data\raw\lahore\pollution\lahore_pollution_hourly.csv


In [38]:
# ==========================================
# Verify saved pollution file
# ==========================================

saved_pollution_df = pd.read_csv(
    pollution_output_path,
    parse_dates=["datetime"]
)

print("Saved file loaded successfully.")
print("Shape:", saved_pollution_df.shape)

display(saved_pollution_df.head())

Saved file loaded successfully.
Shape: (22632, 8)


,datetime,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi
0,2024-01-01 00:00:00,260.8,375.3,4925.0,154.8,51.5,11.0,272.10837
1,2024-01-01 01:00:00,260.8,375.3,4687.0,147.0,41.1,6.0,272.72916
2,2024-01-01 02:00:00,239.3,344.6,4326.0,132.2,32.1,8.0,273.39163
3,2024-01-01 03:00:00,208.6,299.0,3733.0,103.5,26.4,24.0,273.42917
4,2024-01-01 04:00:00,196.0,281.3,3016.0,67.8,22.2,48.0,273.07916


## 10. Validate Weather and Air Quality Timestamps

Both datasets contain hourly observations. Before merging them,
we verify that their timestamps align correctly.

In [40]:
weather_timestamps = set(
    weather_df["datetime"]
)

pollution_timestamps = set(
    pollution_df["datetime"]
)

common_timestamps = (
    weather_timestamps
    .intersection(pollution_timestamps)
)

print("Weather Data rows:", len(weather_df))

print(
    "Weather Data unique timestamps:",
    weather_df["datetime"].nunique()
)

print("Pollution Data rows:", len(pollution_df))

print(
    "Pollution Data unique timestamps:",
    pollution_df["datetime"].nunique()
)

print(
    "Common timestamps:",
    len(common_timestamps)
)

Weather Data rows: 22632
Weather Data unique timestamps: 22632
Pollution Data rows: 22632
Pollution Data unique timestamps: 22632
Common timestamps: 22632


## 11. Merge Weather and Air Quality Data

The weather and air-quality datasets are merged using the common
hourly `datetime` column.

In [41]:
lahore_df = pd.merge(
    weather_df,
    pollution_df,
    on="datetime",
    how="inner"
)

print("Merged dataset shape:", lahore_df.shape)

display(lahore_df.head())

Merged dataset shape: (22632, 15)


,datetime,temperature_2m,relative_humidity_2m,surface_pressure,precipitation,cloud_cover,wind_speed_10m,wind_direction_10m,pm2_5,pm10,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi
0,2024-01-01 00:00:00,6.15,98.969955,992.694580,0.0,100.0,3.415260,161.564957,260.799988,375.299988,4925.0,154.800003,51.500000,11.0,272.108368
1,2024-01-01 01:00:00,6.05,98.287132,992.587891,0.0,100.0,2.276840,161.564957,260.799988,375.299988,4687.0,147.000000,41.099998,6.0,272.729156
2,2024-01-01 02:00:00,6.35,98.971619,992.616333,0.0,100.0,2.189795,260.537750,239.300003,344.600006,4326.0,132.199997,32.099998,8.0,273.391632
3,2024-01-01 03:00:00,6.15,99.655602,992.694580,0.0,100.0,3.096837,305.537750,208.600006,299.000000,3733.0,103.500000,26.400000,24.0,273.429169
4,2024-01-01 04:00:00,6.55,97.282684,992.538025,0.0,100.0,3.096837,305.537750,196.000000,281.299988,3016.0,67.800003,22.200001,48.0,273.079163


## validating the merged dataset

In [42]:
print("Missing values in merged dataset:")

display(
    lahore_df.isna().sum()
)

Missing values in merged dataset:


datetime                0
temperature_2m          0
relative_humidity_2m    0
surface_pressure        0
precipitation           0
cloud_cover             0
wind_speed_10m          0
wind_direction_10m      0
pm2_5                   0
pm10                    0
carbon_monoxide         0
nitrogen_dioxide        0
sulphur_dioxide         0
ozone                   0
us_aqi                  0
dtype: int64

In [43]:
print(
    "Duplicate timestamps:",
    lahore_df["datetime"].duplicated().sum()
)

Duplicate timestamps: 0


In [44]:
print(
    "Minimum datetime:",
    lahore_df["datetime"].min()
)

print(
    "Maximum datetime:",
    lahore_df["datetime"].max()
)

Minimum datetime: 2024-01-01 00:00:00
Maximum datetime: 2026-07-31 23:00:00


In [45]:
from pathlib import Path

# ==========================================
# Save merged weather + pollution data
# ==========================================

merged_output_path = Path(
    "../data/raw/lahore/lahore_weather_pollution_merged_hourly.csv"
)

# Create directory if it doesn't exist
merged_output_path.parent.mkdir(parents=True, exist_ok=True)

# Save merged dataset
lahore_df.to_csv(
    merged_output_path,
    index=False
)

print("Merged dataset saved successfully.")
print(merged_output_path)

Merged dataset saved successfully.
..\data\raw\lahore\lahore_weather_pollution_merged_hourly.csv
